<a href="https://colab.research.google.com/github/massimilianogasparini-author/creative-loop-dynamics/blob/main/mad_data_laundering_validator_creative_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importazione delle librerie necessarie
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configurazione dello stile visivo per il grafico
sns.set_theme(style="white")

def calculate_and_visualize_cs(lz, embedding, perplexity, correlation, lambda_weight):
    """
    Funzione di callback che calcola il Creativity Score e aggiorna la UI
    ogni volta che uno slider viene mosso dall'utente.
    """
    # 1. Calcolo del Punteggio Base (Segnale)
    # Assumiamo pesi equi w_i = 1/3 per le tre metriche
    base_score = (lz + embedding + perplexity) / 3.0

    # 2. Creazione della Matrice di Covarianza 3x3
    # Sulla diagonale abbiamo la varianza normalizzata (1.0)
    # Fuori diagonale abbiamo la correlazione incrociata tra le metriche
    cov_matrix = np.array([
        [1.0, correlation, correlation],
        [correlation, 1.0, correlation],
        [correlation, correlation, 1.0]
    ])

    # 3. Calcolo della Norma di Frobenius della matrice off-diagonal
    # In una matrice 3x3, ci sono 6 elementi fuori diagonale.
    # Se tutti valgono 'correlation', la somma dei quadrati è 6 * correlation^2
    # La radice quadrata è sqrt(6) * correlation (circa 2.449 * correlation)
    frobenius_norm = np.sqrt(6) * correlation

    # 4. Calcolo della Penalità e del Punteggio Finale
    penalty = lambda_weight * frobenius_norm
    final_cs = max(0.0, base_score - penalty) # Il punteggio non può essere negativo

    # --- RENDERIZZAZIONE DELL'INTERFACCIA ---
    clear_output(wait=True) # Pulisce l'output precedente per evitare duplicati

    # Stampa dei risultati analitici
    print("="*50)
    print(" ANALISI IN TEMPO REALE: CREATIVITY SCORE (Cs)")
    print("="*50)
    print(f"Punteggio Base (Segnale Medio):  {base_score:.3f}")
    print(f"Penalità (Norma di Frobenius):  -{penalty:.3f}")
    print("-" * 50)
    print(f"PUNTEGGIO FINALE Cs:             {final_cs:.3f}")
    print("="*50)

    # Avviso termodinamico in caso di collasso
    if final_cs <= 0.0:
        print("\n⚠️ ATTENZIONE: COLLASSO AUTÒFAGO RILEVATO!")
        print("La ridondanza informativa ha annullato la validità del dato.")
        print("Il campione viene scartato per prevenire l'inquinamento del dataset.\n")
    else:
        print("\n✅ Dato validato. Procedere con l'iniezione (flusso φ).\n")

    # Rendering della Matrice di Covarianza
    fig, ax = plt.subplots(figsize=(6, 5))

    # Creiamo una maschera per isolare visivamente la diagonale (che è buona)
    # dagli elementi fuori diagonale (che rappresentano la ridondanza tossica)
    cmap = sns.light_palette("red", as_cmap=True)

    sns.heatmap(cov_matrix, annot=True, fmt=".2f", cmap=cmap, vmin=0, vmax=1,
                xticklabels=['LZ', 'Embedding', 'Perplexity'],
                yticklabels=['LZ', 'Embedding', 'Perplexity'],
                linewidths=.5, ax=ax, cbar_kws={'label': 'Livello di Ridondanza'})

    ax.set_title("Matrice di Covarianza (Σ)\n(L'intensità del rosso indica la penalità)", fontsize=12, pad=15)
    plt.tight_layout()
    plt.show()

# =====================================================================
# DEFINIZIONE DEGLI SLIDER INTERATTIVI (ipywidgets)
# =====================================================================
style = {'description_width': 'initial'}
layout = widgets.Layout(width='500px')

lz_slider = widgets.FloatSlider(value=0.8, min=0, max=1.0, step=0.01, description='Complessità LZ:', style=style, layout=layout)
embed_slider = widgets.FloatSlider(value=0.8, min=0, max=1.0, step=0.01, description='Dist. Embedding:', style=style, layout=layout)
perp_slider = widgets.FloatSlider(value=0.8, min=0, max=1.0, step=0.01, description='Perplexity:', style=style, layout=layout)

corr_slider = widgets.FloatSlider(value=0.1, min=0, max=1.0, step=0.01, description='Correlazione (Ridondanza):', style=style, layout=layout)
lambda_slider = widgets.FloatSlider(value=0.3, min=0, max=1.0, step=0.01, description='Peso Penalità (λ):', style=style, layout=layout)

# Creazione dell'interfaccia utente impilando i widget
ui = widgets.VBox([
    widgets.HTML("<b>Metriche Proxy (Il Segnale)</b>"),
    lz_slider, embed_slider, perp_slider,
    widgets.HTML("<br><b>Parametri di Penalizzazione (Il Rumore/Ridondanza)</b>"),
    corr_slider, lambda_slider
])

# Collegamento degli slider alla funzione di calcolo
out = widgets.interactive_output(calculate_and_visualize_cs, {
    'lz': lz_slider,
    'embedding': embed_slider,
    'perplexity': perp_slider,
    'correlation': corr_slider,
    'lambda_weight': lambda_slider
})

# Mostra l'interfaccia completa
display(ui, out)